# FigureVault on Google Colab
This notebook sets up the environment to run the FigureVault pipeline on Google Colab.
**Before running:** Make sure to go to `Runtime` -> `Change runtime type` and select **T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Note: Zip your figureVault folder and upload it to your Google Drive first.
# Change the path below to match where you uploaded your zip file.
!unzip "/content/drive/MyDrive/figureVault.zip" -d /content/figureVault
%cd /content/figureVault

In [ ]:
# Install python dependencies from requirements.txt
!pip install -r requirements.txt

# Install system dependencies for computer vision libraries and Java (for PDFFigures2)
!apt-get update
!apt-get install -y libgl1-mesa-glx pciutils lshw zstd default-jdk

### Build PDFFigures2 from source
PDFFigures2 is required for high-accuracy bounding box extraction. It must be built from source using the Scala Build Tool (`sbt`).

In [ ]:
# Install sbt (Scala Build Tool)
!apt-get install apt-transport-https curl gnupg -yqq
!echo "deb https://repo.scala-sbt.org/scalasbt/debian all main" | sudo tee /etc/apt/sources.list.d/sbt.list
!echo "deb https://repo.scala-sbt.org/scalasbt/debian /" | sudo tee /etc/apt/sources.list.d/sbt_old.list
!curl -sL "https://keyserver.ubuntu.com/pks/lookup?op=get&search=0x2EE0EA64E40A89B84B2DF73499E82A75642AC823" | sudo -H gpg --no-default-keyring --keyring gnupg-ring:/etc/apt/trusted.gpg.d/scalasbt-release.gpg --import
!chmod 644 /etc/apt/trusted.gpg.d/scalasbt-release.gpg
!apt-get update
!apt-get install sbt -yqq

# Clone and build PDFFigures2
!git clone https://github.com/allenai/pdffigures2.git /content/pdffigures2
%cd /content/pdffigures2
!sbt assembly

# Copy the built JAR back to FigureVault
!mkdir -p /content/figureVault/bin
!cp pdffigures2.jar /content/figureVault/bin/pdffigures2.jar
%cd /content/figureVault
print("✅ PDFFigures2 successfully built and installed!")

In [ ]:
# Install Ollama for local LLM inference
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

# Start the Ollama server in the background
process = subprocess.Popen(['ollama', 'serve'])
print("Ollama server started in background.")
time.sleep(5)  # Wait for the server to initialize

# Pull the Gemma 4 model
!ollama pull gemma4:e4b

In [ ]:
# Run tests to verify the setup
!python -m pytest tests/

In [ ]:
# Run the main FigureVault pipeline (View the help menu to see commands)
!python main.py --help

# Example usage:
# !python main.py process /content/drive/MyDrive/some_paper.pdf

### Optional: Run Streamlit UI
The cell below uses Cloudflare tunnels instead of localtunnel, which fixes the 'Failed to fetch dynamically imported module' error by skipping the warning screen that localtunnel forces on background JS requests.

In [ ]:
# Start Streamlit in the background
import os
os.system("streamlit run ui/app.py &>/content/logs.txt &")

# Download and install Cloudflare tunnel
!wget -q -O - https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 > /tmp/cloudflared
!chmod +x /tmp/cloudflared

# Expose Streamlit port 8501. Look for the link ending in .trycloudflare.com in the output!
!/tmp/cloudflared tunnel --url http://localhost:8501